# LAB004–012 — química y CO₂ con `theta_natural_full` congelado

Comparación diagnóstica sin fitting: se conservan los 17 parámetros cinéticos de `theta_natural_full.csv` y los nueve parámetros de la capa CO₂ del estado `C_full_theta_refit`. Los escenarios cambian únicamente el balance de volumen y la mezcla del pulso. LAB004–012 son replay de química usada en calibración; LAB012 es holdout **solo para la capa CO₂**. LAB009 carece de CO₂ calibrable y figura únicamente en química.

**Supuestos configurables, aún no comprobados:** 65 mL netos por fila con E homologado finito; la extracción en t=0 sí cuenta; 20 mL de solución nutritiva es un ejemplo, no una medición. El pulso del endpoint CO₂ tiene masa fija de 280 mg YAN (históricamente 0,14 kg/m³ en 2 L); la calibración cinética original usó 0,08 kg/m³, discrepancia preexistente que este ensayo no resuelve.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

candidates = (Path.cwd(), Path.cwd() / 'volume_analysis',
              Path.cwd() / 'laboratory_2026' / 'notebooks' / 'volume_analysis',
              Path.cwd() / 'fermentation_model' / 'laboratory_2026' / 'notebooks' / 'volume_analysis')
notebook_dir = next((p for p in candidates if (p / '_volume_analysis.py').is_file()), None)
if notebook_dir is None:
    raise FileNotFoundError('Ejecutar desde el repositorio, notebooks o volume_analysis')
sys.path.insert(0, str(notebook_dir))
import _volume_analysis as va
from shared import run_new_must_glycerol_estimability_doe as kinetic
from laboratory_2026 import run_co2_matrix_cross_validation_2026 as co2

ALCOLYZER_NET_LOSS_ML = 65.0  # hipótesis; no dato individual verificado
ILLUSTRATIVE_FEED_ML = 20.0  # cambiar por volumen medido cuando exista
INCLUDE_T0_SAMPLE = True
SCENARIOS = {
    'actual': (0.0, 0.0),
    'muestras_65': (ALCOLYZER_NET_LOSS_ML, 0.0),
    'nutricion_20': (0.0, ILLUSTRATIVE_FEED_ML),
    'ambos': (ALCOLYZER_NET_LOSS_ML, ILLUSTRATIVE_FEED_ML),
}
batches, tables = va.load_context()
theta, co2_fit = va.load_frozen_parameters()
ledgers = {
    scenario: va.make_ledgers(tables, alcolyzer_ml=sample_ml, feed_ml=feed_ml,
                              include_t0_sample=INCLUDE_T0_SAMPLE)
    for scenario, (sample_ml, feed_ml) in SCENARIOS.items()
}
labs = sorted(batches)
print('Parámetros cinéticos fijos:', len(theta), '| parámetros CO₂ fijos:', len(co2_fit))
print('Escenarios:', SCENARIOS)

## 1. Comprobación de referencia y predicciones químicas

La variante de volumen cero debe reproducir el simulador vigente. Las muestras observadas se comparan con el estado **anterior** a la extracción; una muestra bien mezclada no crea saltos de concentración. El pulso mezcla la masa fija de YAN con el volumen posterior y diluye los estados que no entran con el stock.

In [ ]:
chemistry_sim = {}
reference_difference = []
for lab, batch in batches.items():
    horizon = float(batch.time[-1])
    grid = np.unique(np.r_[np.arange(0.0, horizon + 1.0, 1.0), batch.time,
                           *[ledger[lab].time_h.to_numpy(float) for ledger in ledgers.values()]])
    grid = grid[(grid >= 0) & (grid <= horizon + 1e-9)]
    reference = kinetic.simulate(batch, theta, grid)
    assert reference is not None
    zero_volume = va.simulate_with_volume(batch, theta, grid, ledgers['actual'][lab])
    delta = np.max(np.abs(reference.to_numpy() - zero_volume.to_numpy()))
    reference_difference.append(delta)
    chemistry_sim[lab] = {'actual': reference}
    for scenario in SCENARIOS:
        if scenario != 'actual':
            chemistry_sim[lab][scenario] = va.simulate_with_volume(
                batch, theta, grid, ledgers[scenario][lab]
            )
assert max(reference_difference) < 1e-8, 'La variante sin cambio de volumen no reproduce el modelo vigente'
print('Máxima diferencia del control de volumen cero:', max(reference_difference))

metric_rows = []
for lab, batch in batches.items():
    for state in kinetic.STATE_NAMES:
        observed = np.asarray(batch.observations[state], dtype=float)
        mask = np.isfinite(observed) & (np.asarray(batch.time) > 0)
        if not mask.any():
            continue
        times = np.asarray(batch.time)[mask]
        sigma = kinetic.sigma_for_state(state, observed[mask])
        for scenario, sim in chemistry_sim[lab].items():
            predicted = sim.loc[times, state].to_numpy(float)
            metric_rows.append({
                'LAB': lab, 'estado': state, 'escenario': scenario, 'n': int(mask.sum()),
                'RMSE': np.sqrt(np.mean((predicted - observed[mask]) ** 2)),
                'RMSE / sigma': np.sqrt(np.mean(((predicted - observed[mask]) / sigma) ** 2)),
                'cambio RMS / sigma vs actual': np.sqrt(np.mean(((predicted - chemistry_sim[lab]['actual'].loc[times, state].to_numpy(float)) / sigma) ** 2)),
            })
chemistry_metrics = pd.DataFrame(metric_rows)
display(chemistry_metrics.query("estado in ['N', 'X', 'G', 'F', 'E']").pivot_table(
    index=['LAB','estado'], columns='escenario', values='RMSE / sigma'
).round(2))

In [ ]:
states = ('N', 'X', 'G', 'F', 'E', 'Gly')
for lab, batch in batches.items():
    fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
    ledger = ledgers['ambos'][lab]
    sample_times = ledger.loc[ledger.event.eq('sample') & ledger.ethanol_observed, 'time_h']
    pulse_times = ledger.loc[ledger.event.eq('nutrition'), 'time_h']
    for ax, state in zip(axes.flat, states):
        for scenario, sim in chemistry_sim[lab].items():
            ax.plot(sim.index, sim[state], lw=1.6, color=va.COLORS[scenario], label=scenario)
        obs = np.asarray(batch.observations[state], dtype=float)
        mask = np.isfinite(obs)
        ax.scatter(np.asarray(batch.time)[mask], obs[mask], color='black', s=18, zorder=5, label='medición')
        for time_h in sample_times:
            ax.axvline(time_h, color='#999999', lw=0.5, alpha=0.28)
        for time_h in pulse_times:
            ax.axvline(time_h, color='#D55E00', lw=1.0, ls='--', alpha=0.7)
        ax.set_title(state)
        ax.set_ylabel(va.STATE_UNITS[state])
        ax.grid(alpha=0.2)
    for ax in axes[-1]:
        ax.set_xlabel('h desde primera muestra química')
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.97), ncol=5, frameon=False)
    fig.suptitle(f'{lab} · química observada y predicha · θ natural fijo', y=1.01)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    plt.show()
    plt.close(fig)

## 2. CO₂ en unidades equivalentes del sensor

La curva observada es el perfil horario **ya filtrado y suavizado** de la corrida SCCM corregida, reconvertido a SCCM equivalente usando sus 2 L originales. Los puntos horarios de SCCM con cero corregido se muestran tenues como contexto, pero no son la curva ajustada. Las predicciones convierten su tasa por litro a caudal total con el V(t) de cada escenario. Se preservan el ajuste CO₂ y la máscara de censura anteriores; todavía no se reconstruye el preprocesamiento con V(t). LAB009 no tiene CO₂ calibrable.

In [ ]:
observations = va.load_co2_observations()
co2_batches = sorted(observations.batch.unique())
print('LAB con CO₂ calibrable:', ', '.join(co2_batches))
print('Excluido del ajuste CO₂:', sorted(set(labs) - set(co2_batches)))
cache_by_scenario = {}
for scenario in SCENARIOS:
    if scenario == 'actual':
        cache_by_scenario[scenario], _ = co2.build_driver_cache(
            batches, {'natural': theta}, observations
        )
    else:
        cache_by_scenario[scenario] = va.build_volume_aware_cache(
            batches, theta, observations, ledgers[scenario]
        )

prediction_rows = []
for lab in co2_batches:
    group = observations[observations.batch.eq(lab)].sort_values('t_h')
    times = group.t_h.to_numpy(float)
    v0 = ledgers['actual'][lab].initial_volume_l.iloc[0]
    observed_sccm = va.observed_processed_sccm(group, v0)
    for scenario in SCENARIOS:
        predicted = va.co2_prediction_sccm(
            cache_by_scenario[scenario][lab], co2_fit, ledgers[scenario][lab], times
        )
        prediction_rows.extend({
            'LAB': lab, 'escenario': scenario, 'time_h': t,
            'observado_sccm_equivalente': y, 'predicho_sccm': p,
            'censurado_original': bool(c),
        } for t, y, p, c in zip(times, observed_sccm, predicted, group.left_censored))
co2_predictions = pd.DataFrame(prediction_rows)
metric_rows = []
for (lab, scenario), group in co2_predictions.groupby(['LAB','escenario']):
    usable = group[~group.censurado_original]
    if usable.empty:
        continue
    error = usable.predicho_sccm - usable.observado_sccm_equivalente
    metric_rows.append({'LAB': lab, 'escenario': scenario, 'n no censurados': len(usable),
                        'RMSE SCCM': np.sqrt(np.mean(error ** 2)), 'sesgo SCCM': error.mean()})
co2_metrics = pd.DataFrame(metric_rows)
display(co2_metrics.pivot(index='LAB', columns='escenario', values='RMSE SCCM').round(3))

In [ ]:
for lab in co2_batches:
    group = observations[observations.batch.eq(lab)].sort_values('t_h')
    fig, (ax, delta_ax) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                       gridspec_kw={'height_ratios': [3, 1]})
    ax.scatter(group.t_h, group.sensor_flow_zero_corrected_sccm, s=6, color='#BBBBBB', alpha=0.45, label='SCCM horario, cero corregido')
    actual = co2_predictions[(co2_predictions.LAB.eq(lab)) & (co2_predictions.escenario.eq('actual'))].sort_values('time_h')
    ax.plot(actual.time_h, actual.observado_sccm_equivalente, color='black', lw=1.5, label='observado filtrado, SCCM eq.')
    for scenario in SCENARIOS:
        subset = co2_predictions[(co2_predictions.LAB.eq(lab)) & (co2_predictions.escenario.eq(scenario))].sort_values('time_h')
        ax.plot(subset.time_h, subset.predicho_sccm, color=va.COLORS[scenario], lw=1.5, label=scenario)
        if scenario != 'actual':
            delta_ax.plot(subset.time_h, subset.predicho_sccm.to_numpy() - actual.predicho_sccm.to_numpy(),
                          color=va.COLORS[scenario], lw=1.3, label=scenario)
    for time_h in ledgers['ambos'][lab].loc[lambda x: x.event.eq('nutrition'), 'time_h']:
        ax.axvline(time_h, color='#D55E00', ls='--', alpha=0.7, lw=1)
    delta_ax.axhline(0, color='black', lw=0.8)
    ax.set_title(lab + (' · holdout de CO₂' if lab == 'LAB012' else ' · replay de CO₂'))
    ax.set_ylabel('Caudal (SCCM)')
    delta_ax.set_ylabel('Δ predicción (SCCM)')
    delta_ax.set_xlabel('h desde primera muestra química')
    ax.grid(alpha=0.2); delta_ax.grid(alpha=0.2)
    ax.legend(loc='best', fontsize=8, ncol=2)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Alcance de esta primera comparación

Estos gráficos muestran sensibilidad con parámetros congelados; no autorizan reemplazar `theta_natural_full.csv` ni interpretar una mejor RMSE como validación. La capa O₂/CO₂ disuelto existente sigue integrando concentraciones sin saltos explícitos de dilución en sus reservorios, y tampoco se cambian headspace, geometría, transferencia de gases o artefactos del sensor. La masa de YAN de 280 mg y los 20 mL de stock son hipótesis separadas: la primera proviene del protocolo que usa la capa CO₂; la segunda es solo ilustrativa. Antes de recalibrar se debe verificar el registro físico y rehacer consistentemente el preprocesamiento de CO₂.